# py-extra-terminal Interactive Testing Guide

This notebook allows you to test the `py-extra-terminal` library features individually. 

**Requirements:**
- Attachmate EXTRA! X-treme installed and running.
- At least one session profile (`.edp`) open.

### 1. Setup and Imports
Run this cell first to initialize the environment.

In [1]:
import sys
import os
import time

# Ensure the src directory is in the path
sys.path.insert(0, os.path.abspath('src'))

from py_extra_terminal import Extra, keys, ExtraError, ScreenTimeout, retry
print("Library imported successfully.")

Library imported successfully.


### 2. Connect to EXTRA!
This cell creates a connection to the EXTRA! System COM object.

In [2]:
try:
    # We initialize it outside a 'with' block for interactive persistence
    ex = Extra(visible=True)
    print(f"Connected to EXTRA! Version: {ex.version}")
except ExtraError as e:
    print(f"Connection failed: {e}")

Connected to EXTRA! Version: 9.3


### 3. List and Select Sessions
See which sessions are open and select the active one.

In [3]:
sessions = ex.sessions
print(f"Open sessions ({len(sessions)}):")
for i, s in enumerate(sessions):
    print(f"[{i}] {s.name} (Connected: {s.connected})")

if sessions:
    sess = ex.active_session
    sc = sess.screen
    print(f"\nSelected session: {sess.name}")
else:
    print("No sessions open. Please open a session in EXTRA! emulator.")

Open sessions (2):
[0] Session2 (Connected: True)
[1] Session1 (Connected: True)

Selected session: Session2


### 4. Reading the Screen
Test different ways to extract text from the terminal.

In [6]:
print("--- Full Screen Dump ---")
for i, line in enumerate(sc.dump()):
    print(f"{i+1:02} | {line}")

print("\n--- Specific Area Reading ---")
# Read row 1, columns 1-20
area_text = sc.get(1, 1, 20)
print(f"Row 1 (1-20): '{area_text}'")

### 5. Searching for Text
Find the exact coordinates of a string on the screen.

In [7]:
search_term = "TROYA"

coord = sc.find(search_term)
if coord:
    print(f"Found '{search_term}' at Row: {coord[0]}, Col: {coord[1]}")
else:
    print(f"'{search_term}' not found on screen.")

### 6. Writing and Cursor Control
Move the cursor and put text into fields.

In [8]:
print(f"Current cursor: {sc.cursor}")

# Chaining example: move to 10,10 and write text
sc.move(10, 10).put("TEST_WRITE", 10, 10)
print(f"New cursor after move: {sc.cursor}")
print(f"Content at (10,10): {sc.get(10, 10, 10)}")

### 7. Displaying ASCII Art
Load and display an ASCII art from a file onto the terminal screen. This demonstrates reading a multi-line text file and writing it line by line.

In [10]:
ascii_art_path = "ascii-art.txt"
try:
    with open(ascii_art_path, "r") as f:
        art_lines = f.readlines()
    
    start_row = 1 # Starting row for the ASCII art
    start_col = 1 # Starting col for the ASCII art
    sc.clear_area(start_row, start_col, start_row + len(art_lines), sc.cols) # Clear area before writing
    
    print(f"\nDisplaying ASCII art from {ascii_art_path} at ({start_row},{start_col})...")
    for i, line in enumerate(art_lines):
        # Ensure line fits within terminal width
        display_line = line.rstrip()[:sc.cols - start_col + 1]
        sc.put(display_line, start_row + i, start_col)
        time.sleep(0.1)
    sc.wait_quiet(ms=500)
    print("ASCII art displayed.")
except FileNotFoundError:
    print(f"Error: ASCII art file \'{ascii_art_path}\' not found.")
except Exception as e:
    print(f"Error displaying ASCII art: {e}")


Displaying ASCII art from ascii-art.txt at (1,1)...
ASCII art displayed.


### 8. Synchronization (Waiters)
Wait for host quietness or specific strings.

In [9]:
print("Waiting for host to be quiet...")
sc.wait_quiet()
print("Host is quiet.")

print("\nWaiting for 'READY' or 'ERROR' (2 sec timeout)...")
try:
    found = sc.wait_for_any(["READY", "ERROR"], timeout=2.0)
    print(f"Detected: {found}")
except ScreenTimeout:
    print("Neither found within timeout.")

### 9. OIA Lock Status
Check if the keyboard is currently locked by the host.

In [10]:
print(f"Is keyboard locked? {sc.is_locked}")
print(f"Is screen updating? {sc.updating}")
print(f"XStatus value: {sc.x_status}")

### 10. Special Keys
Test all available terminal-specific keys. 
**Caution:** Sending some keys like PF3 might exit your terminal screen.

In [11]:
print(f"Available keys in the 'keys' module:")
all_keys = [k for k in dir(keys) if not k.startswith('__')]
print(all_keys)

# Chaining example: Send TAB then RESET
print("\nSending TAB and then RESET...")
sc.send(keys.TAB).wait_quiet(ms=200).send(keys.RESET)
print("Done.")

time.sleep(5)

# Chaining example 2: Send ENTER then CLEAR
print("\nSending ENTER and then CLEAR...")
sc.put("asdasdasd",2,15)
sc.send(keys.ENTER).wait_quiet(ms=200)

screen_content = sc.get(1,15,80)
print(f"\nScreen content : {screen_content}")

sc.send(keys.CLEAR)
print("Done.")

In [12]:
sc.send(keys.ENTER).wait_quiet(ms=200).send(keys.CLEAR)

### 11. Cleanup
Disconnect the COM session.

In [13]:
ex.close()
print("Disconnected from EXTRA! System.")